# FastAPI — API for Model Serving

In [ ]:
# Day 34 — FastAPI
# We've trained models (Day 27-29), saved them (Day 30, 32),
# versioned data (Day 33). But HOW do other services USE the model?
#
# Answer: through an API (Application Programming Interface)
#
# Analogy:
# The trained model = a chef who can cook amazing dishes
# FastAPI = the restaurant (takes orders, delivers food to customers)
# Customers = other services that need predictions
#
# Without API: you run predict.py manually on your laptop
# With API:    any service can send data → get predictions automatically
#
# FastAPI is the #1 Python framework for building APIs because:
#   - Fast (built on Starlette + Pydantic)
#   - Auto-generates documentation (Swagger UI)
#   - Type-safe (catches errors before runtime)
#   - Async support (handles many requests at once)
#   - Used by Microsoft, Netflix, Uber for ML serving
#
# Today we'll cover:
#   1. First API (GET endpoints)
#   2. Path & query parameters
#   3. POST requests & Pydantic models
#   4. Serving an ML model (the main goal!)
#   5. Error handling
#   6. Auto-generated docs (Swagger UI)
#
# Install: pip install fastapi uvicorn
#
# NOTE: FastAPI runs as a SERVER, not inside Jupyter cells.
# We'll write .py files and run them from the terminal.

### Your First FastAPI App

In [ ]:
# Let's create our first API!
# We'll write the code to a .py file and explain each line

import os
os.makedirs('fastapi_demo', exist_ok=True)

code = '''from fastapi import FastAPI

# Create the app (like Flask's app = Flask(__name__))
app = FastAPI(
    title="Server Health API",
    description="API for monitoring server health",
    version="1.0.0"
)

# Define a GET endpoint at /
# When someone visits http://localhost:8000/, they get this response
@app.get("/")
def root():
    return {"message": "Server Health API is running!", "status": "healthy"}

# Another GET endpoint
@app.get("/health")
def health_check():
    return {"status": "ok", "api_version": "1.0.0"}
'''

with open('fastapi_demo/app_basic.py', 'w') as f:
    f.write(code)

print('fastapi_demo/app_basic.py created!')
print()
print('To run this API:')
print('  cd fastapi_demo')
print('  uvicorn app_basic:app --reload')
print()
print('Then open:')
print('  http://localhost:8000          → root endpoint')
print('  http://localhost:8000/health   → health check')
print('  http://localhost:8000/docs     → Swagger UI (auto-generated!)')
print()
print('The code:')
print(code)

### Path & Query Parameters

In [ ]:
# Path parameters: part of the URL path
#   /server/web-01        → server_id = "web-01"
#   /server/db-03         → server_id = "db-03"
#
# Query parameters: after the ? in the URL
#   /servers?region=eastus&status=critical
#     → region = "eastus", status = "critical"

code = '''from fastapi import FastAPI
from typing import Optional

app = FastAPI(title="Server Monitoring API")

# Fake server data (in real life, this comes from a database)
servers = {
    "web-01": {"cpu": 75, "memory": 60, "region": "eastus", "status": "healthy"},
    "web-02": {"cpu": 92, "memory": 88, "region": "eastus", "status": "critical"},
    "db-01":  {"cpu": 45, "memory": 70, "region": "westeurope", "status": "healthy"},
    "db-02":  {"cpu": 85, "memory": 90, "region": "westeurope", "status": "warning"},
    "api-01": {"cpu": 30, "memory": 40, "region": "eastus", "status": "healthy"},
}

# PATH parameter — get a specific server
@app.get("/server/{server_id}")
def get_server(server_id: str):
    if server_id in servers:
        return {"server_id": server_id, **servers[server_id]}
    return {"error": f"Server {server_id} not found"}

# QUERY parameters — filter servers
@app.get("/servers")
def list_servers(region: Optional[str] = None, status: Optional[str] = None):
    result = []
    for name, info in servers.items():
        if region and info["region"] != region:
            continue
        if status and info["status"] != status:
            continue
        result.append({"server_id": name, **info})
    return {"count": len(result), "servers": result}

# Both combined
@app.get("/server/{server_id}/metric/{metric_name}")
def get_metric(server_id: str, metric_name: str):
    if server_id not in servers:
        return {"error": f"Server {server_id} not found"}
    if metric_name not in servers[server_id]:
        return {"error": f"Metric {metric_name} not found"}
    return {"server_id": server_id, "metric": metric_name, "value": servers[server_id][metric_name]}
'''

with open('fastapi_demo/app_params.py', 'w') as f:
    f.write(code)

print('Examples of calling this API:')
print()
print('  GET /server/web-01')
print('    → {"server_id": "web-01", "cpu": 75, "memory": 60, ...}')
print()
print('  GET /servers?region=eastus')
print('    → {"count": 3, "servers": [web-01, web-02, api-01]}')
print()
print('  GET /servers?status=critical')
print('    → {"count": 1, "servers": [web-02]}')
print()
print('  GET /server/web-01/metric/cpu')
print('    → {"server_id": "web-01", "metric": "cpu", "value": 75}')

### POST Requests & Pydantic Models

In [ ]:
# GET = read data (like viewing a page)
# POST = send data (like submitting a form)
#
# For ML predictions, we use POST because:
#   - We're SENDING feature data to the model
#   - The data can be complex (not just a URL parameter)
#
# Pydantic = data validation library (comes with FastAPI)
# It ensures the data sent to your API is the RIGHT TYPE
#
# Without Pydantic: someone sends cpu="hello" → model crashes
# With Pydantic: FastAPI rejects it with a clear error message

code = '''from fastapi import FastAPI
from pydantic import BaseModel, Field
from typing import List, Optional

app = FastAPI(title="Server Health API")

# Define the INPUT schema (what the client sends)
class ServerMetrics(BaseModel):
    server_id: str = Field(..., example="web-01")
    cpu: float = Field(..., ge=0, le=100, description="CPU usage percentage")
    memory: float = Field(..., ge=0, le=100, description="Memory usage percentage")
    disk: float = Field(..., ge=0, le=100, description="Disk usage percentage")
    error_count: int = Field(..., ge=0, description="Number of errors in last hour")

# Define the OUTPUT schema (what the API returns)
class HealthPrediction(BaseModel):
    server_id: str
    status: str
    confidence: float
    recommendation: str

# Simple rule-based prediction (before we add ML)
@app.post("/predict", response_model=HealthPrediction)
def predict_health(metrics: ServerMetrics):
    # Simple threshold logic (we'll replace with ML model later)
    risk_score = (metrics.cpu * 0.3 + metrics.memory * 0.25 +
                  metrics.disk * 0.15 + metrics.error_count * 0.8)

    if risk_score > 60:
        status = "critical"
        recommendation = "Investigate immediately! Scale up or restart."
        confidence = min(risk_score / 100, 0.99)
    elif risk_score > 40:
        status = "warning"
        recommendation = "Monitor closely. Consider scaling."
        confidence = 0.7
    else:
        status = "healthy"
        recommendation = "No action needed."
        confidence = 0.95

    return HealthPrediction(
        server_id=metrics.server_id,
        status=status,
        confidence=round(confidence, 2),
        recommendation=recommendation
    )

# Batch prediction — predict for MULTIPLE servers at once
@app.post("/predict/batch")
def predict_batch(servers: List[ServerMetrics]):
    results = []
    for server in servers:
        result = predict_health(server)
        results.append(result)
    return {"count": len(results), "predictions": results}
'''

with open('fastapi_demo/app_pydantic.py', 'w') as f:
    f.write(code)

print('POST request example (using curl or Python requests):')
print()
print('curl -X POST http://localhost:8000/predict \\')
print('  -H "Content-Type: application/json" \\')
print('  -d \'{"server_id": "web-01", "cpu": 85, "memory": 90, "disk": 70, "error_count": 25}\'')
print()
print('Pydantic validates automatically:')
print('  cpu="hello"  → 422 Error: value is not a valid float')
print('  cpu=150      → 422 Error: value must be <= 100')
print('  (missing cpu) → 422 Error: field required')

### Serving an ML Model

In [ ]:
# THIS is the main goal — serve a REAL ML model through FastAPI
#
# The flow:
#   1. Train model (Day 27-29) → save with joblib (Day 30)
#   2. FastAPI loads the saved model at STARTUP
#   3. Client sends server metrics → API returns prediction
#
# Let's first train and save a model

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import joblib

# Generate training data
np.random.seed(42)
n = 500
X = pd.DataFrame({
    'cpu': np.random.randint(10, 100, n),
    'memory': np.random.randint(15, 95, n),
    'disk': np.random.randint(10, 90, n),
    'error_count': np.random.randint(0, 50, n)
})
score = 0.3*X['cpu'] + 0.25*X['memory'] + 0.15*X['disk'] + 0.8*X['error_count'] + np.random.normal(0, 10, n)
y = (score > 50).astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train and save
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42))
])
pipe.fit(X_train, y_train)

accuracy = pipe.score(X_test, y_test)
print(f'Model trained! Accuracy: {accuracy:.2%}')

# Save the model
joblib.dump(pipe, 'fastapi_demo/server_health_model.pkl')
print('Model saved to fastapi_demo/server_health_model.pkl')

In [ ]:
# Now create the FastAPI app that LOADS and SERVES the model

code = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List
import joblib
import numpy as np
import pandas as pd

# --- Load model at startup (not on every request!) ---
model = joblib.load("server_health_model.pkl")
print("Model loaded successfully!")

app = FastAPI(
    title="Server Health Prediction API",
    description="Predict server health status using ML",
    version="1.0.0"
)

# --- Request/Response schemas ---
class ServerInput(BaseModel):
    cpu: float = Field(..., ge=0, le=100, description="CPU usage %")
    memory: float = Field(..., ge=0, le=100, description="Memory usage %")
    disk: float = Field(..., ge=0, le=100, description="Disk usage %")
    error_count: int = Field(..., ge=0, description="Error count")

    class Config:
        json_schema_extra = {
            "example": {"cpu": 85, "memory": 90, "disk": 70, "error_count": 25}
        }

class PredictionOutput(BaseModel):
    status: str
    probability: float
    features_received: dict

# --- Endpoints ---

@app.get("/")
def root():
    return {"service": "Server Health Prediction", "status": "running"}

# Health check (Kubernetes uses this for liveness probes)
@app.get("/healthz")
def healthz():
    return {"status": "ok"}

# Readiness check (Kubernetes uses this for readiness probes)
@app.get("/readyz")
def readyz():
    try:
        # Verify model can make predictions
        test_input = pd.DataFrame({"cpu": [50], "memory": [50], "disk": [50], "error_count": [5]})
        model.predict(test_input)
        return {"status": "ready", "model_loaded": True}
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"Not ready: {str(e)}")

# Model info endpoint
@app.get("/model/info")
def model_info():
    return {
        "model_type": "RandomForestClassifier",
        "features": ["cpu", "memory", "disk", "error_count"],
        "target": "health_status",
        "classes": {"0": "healthy", "1": "critical"}
    }

# SINGLE prediction
@app.post("/predict", response_model=PredictionOutput)
def predict(server: ServerInput):
    # Convert input to DataFrame (what sklearn expects)
    features = pd.DataFrame([{
        "cpu": server.cpu,
        "memory": server.memory,
        "disk": server.disk,
        "error_count": server.error_count
    }])

    # Predict
    prediction = model.predict(features)[0]
    probability = model.predict_proba(features)[0].max()

    return PredictionOutput(
        status="critical" if prediction == 1 else "healthy",
        probability=round(float(probability), 4),
        features_received=server.model_dump()
    )

# BATCH prediction
@app.post("/predict/batch")
def predict_batch(servers: List[ServerInput]):
    # Convert all inputs to a single DataFrame
    features = pd.DataFrame([s.model_dump() for s in servers])

    # Predict all at once (much faster than one by one)
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)

    results = []
    for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
        results.append({
            "index": i,
            "status": "critical" if pred == 1 else "healthy",
            "probability": round(float(prob.max()), 4)
        })

    return {
        "count": len(results),
        "predictions": results
    }
'''

with open('fastapi_demo/app_ml.py', 'w') as f:
    f.write(code)

print('ML serving API created: fastapi_demo/app_ml.py')
print()
print('To run:')
print('  cd fastapi_demo')
print('  uvicorn app_ml:app --reload --port 8000')
print()
print('Then test with curl:')
print('  curl http://localhost:8000/model/info')
print('  curl -X POST http://localhost:8000/predict \\')
print('    -H "Content-Type: application/json" \\')
print('    -d \'{"cpu": 85, "memory": 90, "disk": 70, "error_count": 25}\'')

### Testing the API with Python requests

In [ ]:
# In production, other services call your API using HTTP requests
# In Python, we use the 'requests' library
#
# NOTE: The API must be RUNNING in a separate terminal for this to work
# If you haven't started the API, these examples show what the calls look like

# Example: How to call the API from Python
test_code = '''
import requests

BASE_URL = "http://localhost:8000"

# Test 1: Health check
response = requests.get(f"{BASE_URL}/healthz")
print(f"Health: {response.json()}")

# Test 2: Single prediction
server_data = {
    "cpu": 85,
    "memory": 90,
    "disk": 70,
    "error_count": 25
}
response = requests.post(f"{BASE_URL}/predict", json=server_data)
print(f"Prediction: {response.json()}")

# Test 3: Batch prediction
batch_data = [
    {"cpu": 85, "memory": 90, "disk": 70, "error_count": 25},
    {"cpu": 30, "memory": 25, "disk": 20, "error_count": 2},
    {"cpu": 95, "memory": 95, "disk": 85, "error_count": 40},
]
response = requests.post(f"{BASE_URL}/predict/batch", json=batch_data)
print(f"Batch: {response.json()}")
'''

with open('fastapi_demo/test_api.py', 'w') as f:
    f.write(test_code)

print('Test script created: fastapi_demo/test_api.py')
print()
print('How to test:')
print('  Terminal 1: cd fastapi_demo && uvicorn app_ml:app --reload')
print('  Terminal 2: cd fastapi_demo && python test_api.py')
print()
print('Or just open http://localhost:8000/docs for Swagger UI!')

### Error Handling & HTTPException

In [ ]:
# Good APIs handle errors gracefully
# FastAPI uses HTTP status codes:
#   200 = OK (success)
#   201 = Created
#   400 = Bad Request (client sent invalid data)
#   404 = Not Found
#   422 = Validation Error (Pydantic catches wrong types)
#   500 = Internal Server Error (something broke)
#   503 = Service Unavailable (model not loaded)

error_handling_code = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

app = FastAPI(title="Server API with Error Handling")

servers_db = {
    "web-01": {"cpu": 75, "status": "healthy"},
    "db-01": {"cpu": 85, "status": "warning"},
}

class ServerUpdate(BaseModel):
    cpu: float = Field(ge=0, le=100)
    status: str = Field(pattern="^(healthy|warning|critical)$")

# 404 — Not Found
@app.get("/server/{server_id}")
def get_server(server_id: str):
    if server_id not in servers_db:
        raise HTTPException(
            status_code=404,
            detail=f"Server {server_id} not found. Available: {list(servers_db.keys())}"
        )
    return {"server_id": server_id, **servers_db[server_id]}

# 400 — Bad Request (custom validation)
@app.put("/server/{server_id}")
def update_server(server_id: str, update: ServerUpdate):
    if server_id not in servers_db:
        raise HTTPException(status_code=404, detail="Server not found")

    # Custom validation: can't go from healthy to critical without warning
    current = servers_db[server_id]["status"]
    if current == "healthy" and update.status == "critical":
        raise HTTPException(
            status_code=400,
            detail="Cannot jump from healthy to critical. Use warning first."
        )

    servers_db[server_id] = update.model_dump()
    return {"message": f"Server {server_id} updated", **servers_db[server_id]}
'''

with open('fastapi_demo/app_errors.py', 'w') as f:
    f.write(error_handling_code)

print('Error handling patterns:')
print()
print('  GET /server/web-01      → 200 OK, returns server data')
print('  GET /server/unknown     → 404 Not Found, clear error message')
print('  POST /predict cpu="hi"  → 422 Validation Error (Pydantic auto)')
print('  PUT /server/web-01      → 400 Bad Request (custom logic)')

### Swagger UI — Auto-Generated Documentation

In [ ]:
# One of FastAPI's BEST features: automatic API documentation
#
# When you run your FastAPI app, these URLs are automatically created:
#   http://localhost:8000/docs     → Swagger UI (interactive!)
#   http://localhost:8000/redoc    → ReDoc (alternative, read-only)
#   http://localhost:8000/openapi.json → OpenAPI spec (machine-readable)
#
# Swagger UI lets you:
#   - See all endpoints and their parameters
#   - Try out endpoints directly from the browser
#   - See request/response schemas
#   - No need for Postman or curl for testing!
#
# The documentation is generated from:
#   - Your function names and docstrings
#   - Pydantic model fields and descriptions
#   - The Field() examples you provide

print('FastAPI Auto-Documentation')
print()
print('When you start the server (uvicorn app_ml:app --reload):')
print()
print('  /docs    → Swagger UI')
print('    - Interactive API explorer')
print('    - Click any endpoint → Try it out → Execute')
print('    - Shows request body schema with examples')
print('    - Shows response format')
print()
print('  /redoc   → ReDoc')
print('    - Clean, read-only documentation')
print('    - Good for sharing with API consumers')
print()
print('This is why FastAPI is preferred over Flask for ML serving:')
print('  Flask: you write documentation manually')
print('  FastAPI: documentation writes itself!')

### Complete Production-Ready API

In [ ]:
# Let's create a COMPLETE production-ready API
# This is what you'd actually deploy to Kubernetes

production_code = '''from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field
from typing import List, Optional
import joblib
import pandas as pd
import numpy as np
from datetime import datetime
import logging

# --- Setup logging ---
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("server-health-api")

# --- Load model ---
try:
    model = joblib.load("server_health_model.pkl")
    MODEL_LOADED = True
    logger.info("Model loaded successfully")
except Exception as e:
    MODEL_LOADED = False
    logger.error(f"Failed to load model: {e}")

# --- App ---
app = FastAPI(
    title="Server Health Prediction API",
    description="ML-powered server health monitoring and prediction",
    version="2.0.0",
    docs_url="/docs",
    redoc_url="/redoc"
)

# --- Track predictions for monitoring ---
prediction_log = []

# --- Schemas ---
class ServerInput(BaseModel):
    server_id: Optional[str] = "unknown"
    cpu: float = Field(..., ge=0, le=100)
    memory: float = Field(..., ge=0, le=100)
    disk: float = Field(..., ge=0, le=100)
    error_count: int = Field(..., ge=0)

class PredictionResponse(BaseModel):
    server_id: str
    prediction: str
    confidence: float
    timestamp: str

# --- Endpoints ---

@app.get("/")
def root():
    return {
        "service": "Server Health Prediction API",
        "version": "2.0.0",
        "model_loaded": MODEL_LOADED,
        "endpoints": ["/predict", "/predict/batch", "/model/info", "/model/stats"]
    }

# Kubernetes liveness probe
@app.get("/healthz")
def healthz():
    return {"status": "alive"}

# Kubernetes readiness probe
@app.get("/readyz")
def readyz():
    if not MODEL_LOADED:
        raise HTTPException(status_code=503, detail="Model not loaded")
    return {"status": "ready"}

@app.get("/model/info")
def model_info():
    return {
        "model_type": type(model).__name__ if MODEL_LOADED else "not loaded",
        "features": ["cpu", "memory", "disk", "error_count"],
        "target": "health_status (0=healthy, 1=critical)",
        "total_predictions": len(prediction_log)
    }

@app.get("/model/stats")
def model_stats():
    if not prediction_log:
        return {"message": "No predictions yet"}
    predictions = [p["prediction"] for p in prediction_log]
    return {
        "total_predictions": len(predictions),
        "healthy_count": predictions.count("healthy"),
        "critical_count": predictions.count("critical"),
        "critical_rate": round(predictions.count("critical") / len(predictions), 4)
    }

@app.post("/predict", response_model=PredictionResponse)
def predict(server: ServerInput):
    if not MODEL_LOADED:
        raise HTTPException(status_code=503, detail="Model not available")

    features = pd.DataFrame([{
        "cpu": server.cpu,
        "memory": server.memory,
        "disk": server.disk,
        "error_count": server.error_count
    }])

    pred = model.predict(features)[0]
    prob = model.predict_proba(features)[0].max()

    result = PredictionResponse(
        server_id=server.server_id,
        prediction="critical" if pred == 1 else "healthy",
        confidence=round(float(prob), 4),
        timestamp=datetime.now().isoformat()
    )

    # Log the prediction
    prediction_log.append(result.model_dump())
    logger.info(f"Prediction: {server.server_id} -> {result.prediction}")

    return result

@app.post("/predict/batch")
def predict_batch(servers: List[ServerInput]):
    if not MODEL_LOADED:
        raise HTTPException(status_code=503, detail="Model not available")

    features = pd.DataFrame([s.model_dump(exclude={"server_id"}) for s in servers])
    predictions = model.predict(features)
    probabilities = model.predict_proba(features)

    results = []
    for i, (server, pred, prob) in enumerate(zip(servers, predictions, probabilities)):
        result = {
            "server_id": server.server_id,
            "prediction": "critical" if pred == 1 else "healthy",
            "confidence": round(float(prob.max()), 4)
        }
        results.append(result)
        prediction_log.append(result)

    return {"count": len(results), "predictions": results}
'''

with open('fastapi_demo/app_production.py', 'w') as f:
    f.write(production_code)

print('Production API created: fastapi_demo/app_production.py')
print()
print('This API includes:')
print('  ✅ ML model loading with error handling')
print('  ✅ Kubernetes health probes (/healthz, /readyz)')
print('  ✅ Model info endpoint')
print('  ✅ Prediction statistics tracking')
print('  ✅ Single and batch predictions')
print('  ✅ Structured logging')
print('  ✅ Pydantic validation')
print('  ✅ Auto-generated Swagger docs')

In [ ]:
# Practice Questions:

# 1. Create a FastAPI app with:
#    GET /  → welcome message
#    GET /server/{id}  → return fake server metrics
#    POST /alert  → accept alert data (server_id, level, message)
#    Run it with uvicorn and test from /docs

# 2. Add error handling to the predict endpoint:
#    - What if the model file doesn't exist?
#    - What if the model prediction fails (bad data)?
#    - Return appropriate HTTP status codes

# 3. Create a /predict/explain endpoint that returns:
#    - The prediction (healthy/critical)
#    - Feature importance (which feature mattered most)
#    - A human-readable explanation

# 4. Write a test script that:
#    - Sends 10 random server metrics to /predict
#    - Checks /model/stats to see the results
#    - Prints a summary of healthy vs critical

### DevOps / AI-MLOps Practice

In [ ]:
# FastAPI is the DEPLOYMENT layer of MLOps — where models meet the real world

# Exercise 1: Multi-Model API
# Create a FastAPI app that serves MULTIPLE models:
#   POST /predict/health   → uses the server health model
#   POST /predict/latency  → uses a latency prediction model
#   GET /models            → list all loaded models and their versions
# Load each model from a separate .pkl file at startup

# Exercise 2: A/B Testing Endpoint
# Create a /predict/ab endpoint that:
#   - Loads 2 versions of the same model (v1 and v2)
#   - Randomly routes 50% traffic to v1, 50% to v2
#   - Logs which model was used in the response
#   - GET /ab/stats shows performance comparison
# This is how real A/B testing works in production ML!

# Exercise 3: Dockerize the API
# Create a Dockerfile for the prediction API:
#   FROM python:3.11-slim
#   WORKDIR /app
#   COPY requirements.txt .
#   RUN pip install -r requirements.txt
#   COPY . .
#   EXPOSE 8000
#   CMD ["uvicorn", "app_production:app", "--host", "0.0.0.0", "--port", "8000"]
# Write the Dockerfile and requirements.txt to fastapi_demo/

# Why this matters for me as a DevOps/MLOps engineer:
# 1. FastAPI is what I deploy to Kubernetes for model serving
# 2. /healthz and /readyz are CRITICAL for K8s liveness/readiness probes
# 3. The API is the interface between data scientists and production
# 4. Docker + FastAPI + K8s = the standard ML deployment stack
# 5. MLflow Registry → Load model → FastAPI serves → K8s scales

# Clean up
import shutil
if os.path.exists('fastapi_demo'):
    shutil.rmtree('fastapi_demo')
    print('Demo files cleaned up!')